# Unidad 11: Ingesta con Auto Loader, Schema Evolution y Cloud Files

**Tecnicatura en Datos – Apache Spark (Databricks)**  
Notebook de práctica — `spark` y `dbutils` ya están disponibles en el entorno

---

## Contenido

1. Configuración inicial y generación de datos de ejemplo (schema evolution v1/v2/v3)
2. Auto Loader con CSV (clientes) — detección de columnas nuevas
3. Auto Loader con JSON (ventas) — particionado por año/mes
4. Auto Loader con Parquet (productos) — formato columnar de producción
5. Monitoreo y métricas de streams activos
6. `foreachBatch`: UPSERT (MERGE) y deduplicación de micro-lotes
7. Resumen y conclusiones

**Objetivo:** Implementar pipelines de ingesta con Auto Loader, demostrando schema evolution en diferentes formatos de archivo (CSV, JSON, Parquet).

---

## 1. Configuración inicial

Creamos datos de ejemplo con distintas versiones de schema para simular la llegada incremental de archivos con columnas nuevas.

In [0]:
# Configuración de rutas
import os
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, ccurrent_timestamp, input_file_name
# Rutas locales para crear archivos de ejemplo
base_path_local = "/Workspace/Users/seorozco@gmail.com/procesamiento_distribuido/notebook/datos/autoloader"

# Rutas en Volumes para ingesta
base_path_volumes = "/Volumes/workspace/default/tempo/autoloader"
checkpoint_path = "/Volumes/workspace/default/checkpoint/autoloder"
schema_path = "/Volumes/workspace/default/checkpoint/schemas"

# Crear carpetas locales si no existen
for carpeta in ["clientes", "ventas", "productos"]:
    os.makedirs(f"{base_path_local}/{carpeta}", exist_ok=True)

print("✅ Rutas configuradas")
print(f"Datos locales: {base_path_local}")
print(f"Ingesta desde: {base_path_volumes}")
print(f"Checkpoints en: {checkpoint_path}")

In [0]:
# Crear archivos CSV - Clientes (con schema evolution)

# VERSIÓN 1: Schema básico (id, nombre, email)
csv_v1 = """id,nombre,email
1,Ana García,ana@example.com
2,Luis Martínez,luis@example.com
3,María López,maria@example.com
4,Carlos Ruiz,carlos@example.com
5,Laura Torres,laura@example.com"""

with open(f"{base_path_local}/clientes/clientes_v1.csv", "w") as f:
    f.write(csv_v1)

print("✅ Creado clientes_v1.csv (3 columnas: id, nombre, email)")

# VERSIÓN 2: Schema con columna nueva (telefono)
csv_v2 = """id,nombre,email,telefono
6,Pedro Sánchez,pedro@example.com,555-0001
7,Sofia Morales,sofia@example.com,555-0002
8,Diego Castro,diego@example.com,555-0003
9,Carmen Silva,carmen@example.com,555-0004
10,Roberto Vega,roberto@example.com,555-0005"""

with open(f"{base_path_local}/clientes/clientes_v2.csv", "w") as f:
    f.write(csv_v2)

print("✅ Creado clientes_v2.csv (4 columnas: + telefono) - SCHEMA EVOLUTION")

# VERSIÓN 3: Schema con otra columna nueva (ciudad)
csv_v3 = """id,nombre,email,telefono,ciudad
11,Isabel Romero,isabel@example.com,555-0006,Madrid
12,Miguel Ángel,miguel@example.com,555-0007,Barcelona
13,Patricia Núñez,patricia@example.com,555-0008,Valencia
14,Javier Ortiz,javier@example.com,555-0009,Sevilla
15,Elena Fernández,elena@example.com,555-0010,Bilbao"""

with open(f"{base_path_local}/clientes/clientes_v3.csv", "w") as f:
    f.write(csv_v3)

print("✅ Creado clientes_v3.csv (5 columnas: + ciudad) - SCHEMA EVOLUTION 2")

In [0]:
# Crear archivos JSON - Ventas (con schema evolution)
import json

# VERSIÓN 1: Schema básico (id, cliente_id, monto, fecha)
ventas_v1 = [
    {"id": 1, "cliente_id": 1, "monto": 150.50, "fecha": "2024-01-15"},
    {"id": 2, "cliente_id": 2, "monto": 320.00, "fecha": "2024-01-16"},
    {"id": 3, "cliente_id": 3, "monto": 89.99, "fecha": "2024-01-17"},
    {"id": 4, "cliente_id": 1, "monto": 450.75, "fecha": "2024-01-18"},
    {"id": 5, "cliente_id": 4, "monto": 210.00, "fecha": "2024-01-19"},
]

with open(f"{base_path_local}/ventas/ventas_v1.json", "w") as f:
    for venta in ventas_v1:
        f.write(json.dumps(venta) + "\n")

print("✅ Creado ventas_v1.json (4 campos: id, cliente_id, monto, fecha)")

# VERSIÓN 2: Schema con campo nuevo (producto)
ventas_v2 = [
    {"id": 6, "cliente_id": 5, "monto": 599.99, "fecha": "2024-01-20", "producto": "Laptop"},
    {"id": 7, "cliente_id": 6, "monto": 125.50, "fecha": "2024-01-21", "producto": "Teclado"},
    {"id": 8, "cliente_id": 7, "monto": 89.99, "fecha": "2024-01-22", "producto": "Mouse"},
    {"id": 9, "cliente_id": 2, "monto": 1200.00, "fecha": "2024-01-23", "producto": "Monitor"},
    {"id": 10, "cliente_id": 8, "monto": 45.00, "fecha": "2024-01-24", "producto": "Cable HDMI"},
]

with open(f"{base_path_local}/ventas/ventas_v2.json", "w") as f:
    for venta in ventas_v2:
        f.write(json.dumps(venta) + "\n")

print("✅ Creado ventas_v2.json (5 campos: + producto) - SCHEMA EVOLUTION")

# VERSIÓN 3: Schema con campos adicionales (descuento, metodo_pago)
ventas_v3 = [
    {"id": 11, "cliente_id": 9, "monto": 350.00, "fecha": "2024-01-25", "producto": "Auriculares", "descuento": 10.0, "metodo_pago": "tarjeta"},
    {"id": 12, "cliente_id": 10, "monto": 799.99, "fecha": "2024-01-26", "producto": "Tablet", "descuento": 0.0, "metodo_pago": "efectivo"},
    {"id": 13, "cliente_id": 3, "monto": 199.99, "fecha": "2024-01-27", "producto": "Webcam", "descuento": 5.0, "metodo_pago": "tarjeta"},
    {"id": 14, "cliente_id": 11, "monto": 450.00, "fecha": "2024-01-28", "producto": "Impresora", "descuento": 15.0, "metodo_pago": "transferencia"},
    {"id": 15, "cliente_id": 12, "monto": 89.50, "fecha": "2024-01-29", "producto": "USB Drive", "descuento": 0.0, "metodo_pago": "tarjeta"},
]

with open(f"{base_path_local}/ventas/ventas_v3.json", "w") as f:
    for venta in ventas_v3:
        f.write(json.dumps(venta) + "\n")

print("✅ Creado ventas_v3.json (7 campos: + descuento, metodo_pago) - SCHEMA EVOLUTION 2")

In [0]:
# Crear archivos Parquet - Productos (con schema evolution)


# VERSIÓN 1: Schema básico (id, nombre, precio)
schema_v1 = StructType([
    StructField("id", IntegerType(), False),
    StructField("nombre", StringType(), True),
    StructField("precio", DoubleType(), True),
])

productos_v1 = [
    Row(id=1, nombre="Laptop Dell", precio=599.99),
    Row(id=2, nombre="Mouse Logitech", precio=25.50),
    Row(id=3, nombre="Teclado Mecánico", precio=89.99),
    Row(id=4, nombre="Monitor Samsung 24\"", precio=199.00),
    Row(id=5, nombre="Auriculares Sony", precio=79.99),
]

df_productos_v1 = spark.createDataFrame(productos_v1, schema_v1)
df_productos_v1.write.mode("overwrite").parquet(f"{base_path_local}/productos/productos_v1.parquet")

print("✅ Creado productos_v1.parquet (3 columnas: id, nombre, precio)")

# VERSIÓN 2: Schema con columna nueva (categoria)
schema_v2 = StructType([
    StructField("id", IntegerType(), False),
    StructField("nombre", StringType(), True),
    StructField("precio", DoubleType(), True),
    StructField("categoria", StringType(), True),
])

productos_v2 = [
    Row(id=6, nombre="Tablet iPad", precio=499.99, categoria="Tablets"),
    Row(id=7, nombre="Cable HDMI", precio=15.99, categoria="Accesorios"),
    Row(id=8, nombre="Webcam HD", precio=59.99, categoria="Periféricos"),
    Row(id=9, nombre="SSD 1TB", precio=129.99, categoria="Almacenamiento"),
    Row(id=10, nombre="Router WiFi", precio=89.99, categoria="Redes"),
]

df_productos_v2 = spark.createDataFrame(productos_v2, schema_v2)
df_productos_v2.write.mode("overwrite").parquet(f"{base_path_local}/productos/productos_v2.parquet")

print("✅ Creado productos_v2.parquet (4 columnas: + categoria) - SCHEMA EVOLUTION")

# VERSIÓN 3: Schema con columnas adicionales (stock, proveedor)
schema_v3 = StructType([
    StructField("id", IntegerType(), False),
    StructField("nombre", StringType(), True),
    StructField("precio", DoubleType(), True),
    StructField("categoria", StringType(), True),
    StructField("stock", IntegerType(), True),
    StructField("proveedor", StringType(), True),
])

productos_v3 = [
    Row(id=11, nombre="Impresora HP", precio=199.99, categoria="Impresión", stock=15, proveedor="HP Inc"),
    Row(id=12, nombre="Scanner Epson", precio=149.99, categoria="Impresión", stock=8, proveedor="Epson"),
    Row(id=13, nombre="USB 64GB", precio=19.99, categoria="Almacenamiento", stock=50, proveedor="SanDisk"),
    Row(id=14, nombre="Hub USB-C", precio=39.99, categoria="Accesorios", stock=25, proveedor="Anker"),
    Row(id=15, nombre="Micrófono USB", precio=79.99, categoria="Audio", stock=12, proveedor="Blue"),
]

df_productos_v3 = spark.createDataFrame(productos_v3, schema_v3)
df_productos_v3.write.mode("overwrite").parquet(f"{base_path_local}/productos/productos_v3.parquet")

print("✅ Creado productos_v3.parquet (6 columnas: + stock, proveedor) - SCHEMA EVOLUTION 2")

In [0]:
# Verificar estructura de archivos creados
print("📁 Estructura de archivos creados:\n")

for carpeta in ["clientes", "ventas", "productos"]:
    print(f"{carpeta}/")
    for item in dbutils.fs.ls(f"file:{base_path_local}/{carpeta}"):
        print(f"  - {item.name}")
    print()

In [0]:
# Crear carpetas en Volumes
dbutils.fs.mkdirs(f"{base_path_volumes}/clientes")
dbutils.fs.mkdirs(f"{base_path_volumes}/ventas")
dbutils.fs.mkdirs(f"{base_path_volumes}/productos")

print("✅ Carpetas creadas en Volumes")
print("")
print("⚠️ IMPORTANTE: Vamos a copiar los archivos GRADUALMENTE para simular llegada incremental")
print("Primero copiaremos solo los archivos v1, luego v2, y finalmente v3")
print("Esto nos permitirá observar schema evolution en acción")

### 📝 Estrategia de ingesta incremental

Para demostrar **schema evolution** de forma realista:

1. **Primera carga (v1)**: Solo archivos con schema inicial
2. **Segunda carga (v2)**: Archivos con columnas adicionales
3. **Tercera carga (v3)**: Archivos con aún más columnas

Auto Loader detectará automáticamente:
- Nuevos archivos
- Cambios en el schema
- Agregará columnas nuevas a la tabla destino
- Llenará con `null` los valores faltantes en registros anteriores

---

In [0]:
dbutils.fs.rm(base_path_volumes, True)
# Copiar solo archivos v1 (schema inicial)
print("\n🚀 CARGA INICIAL - Archivos con schema base (v1)\n")
# CSV - Clientes v1
dbutils.fs.cp(
    f"file:{base_path_local}/clientes/clientes_v1.csv",
    f"{base_path_volumes}/clientes/clientes_v1.csv"
)
print("✅ Copiado clientes_v1.csv")

# JSON - Ventas v1
dbutils.fs.cp(
    f"file:{base_path_local}/ventas/ventas_v1.json",
    f"{base_path_volumes}/ventas/ventas_v1.json"
)
print("✅ Copiado ventas_v1.json")

# Parquet - Productos v1
dbutils.fs.cp(
    f"file:{base_path_local}/productos/productos_v1.parquet",
    f"{base_path_volumes}/productos/productos_v1.parquet",
    recurse=True
)
print("✅ Copiado productos_v1.parquet")

print("\n✅ Archivos v1 listos para ingesta incremental")

---

## 2. Auto Loader con CSV (Clientes)

Vamos a ingestar archivos CSV con Auto Loader y observar schema evolution cuando lleguen archivos con columnas nuevas.

**Schema inicial (v1):** id, nombre, email  
**Schema v2:** + telefono  
**Schema v3:** + ciudad

---

In [0]:
from pyspark.sql.functions import col, current_timestamp, input_file_name

# Configuración para CSV
csv_source = f"{base_path_volumes}/clientes"
csv_checkpoint = f"{checkpoint_path}/clientes_csv"
csv_schema = f"{schema_path}/clientes_csv"

print(f"📁 Fuente: {csv_source}")
print(f"💾 Checkpoint: {csv_checkpoint}")
print(f"📊 Schema: {csv_schema}")

In [0]:
# Limpiar checkpoint y schema previos (solo para demo)
dbutils.fs.rm(csv_checkpoint, True)
dbutils.fs.rm(csv_schema, True)

# Crear tabla Delta si no existe
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

# Limpiar tabla anterior (solo para demo)
spark.sql("DROP TABLE IF EXISTS bronze.clientes")

print("🧹 Limpieza completada - iniciando desde cero")

# Configurar Auto Loader para CSV
df_clientes = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", csv_schema)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # Clave para schema evolution
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    .load(csv_source)
    .withColumn("fecha_ingesta", current_timestamp())
    .withColumn("archivo_origen", col("_metadata.file_path"))
    # .drop("_rescued_data")
)

print("✅ Stream configurado para CSV")
print("\nSchema inferido inicial:")
df_clientes.printSchema()

In [0]:
def leer_escribir_csv():
    df_clientes = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", csv_schema)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # Clave para schema evolution
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .load(csv_source)
        .withColumn("fecha_ingesta", current_timestamp())
        .withColumn("archivo_origen", col("_metadata.file_path"))
    )

    query_clientes = (df_clientes.writeStream
        .format("delta")
        .option("checkpointLocation", csv_checkpoint)
        .option("mergeSchema", "true")  # Permitir evolución de schema en Delta
        .trigger(availableNow=True)  # Procesar datos disponibles (compatible con Serverless)
        .table("bronze.clientes")
    )

In [0]:

leer_escribir_csv()
print("\n✅ Procesamiento inicial completado")

In [0]:
%sql
-- Verificar datos cargados (v1)
SELECT * FROM bronze.clientes ORDER BY id

In [0]:
%sql
-- Ver schema actual de la tabla
DESCRIBE EXTENDED bronze.clientes

In [0]:

# Copiar archivo v2 (con columna "telefono")
dbutils.fs.cp(
    f"file:{base_path_local}/clientes/clientes_v2.csv",
    f"{base_path_volumes}/clientes/clientes_v2.csv"
)

In [0]:
print("\n🔄 SCHEMA EVOLUTION - Agregando archivos con columna nueva\n")

print("✅ Copiado clientes_v2.csv (incluye columna 'telefono')")
query_clientes = leer_escribir_csv()
print("\n✅ Auto Loader detectó y procesó el nuevo schema")

In [0]:
%sql
-- Verificar que se agregó la columna "telefono"
-- Los registros antiguos deben tener NULL en telefono
SELECT 
    *
FROM bronze.clientes 
ORDER BY id

In [0]:
# Copiar archivo v3 (con columna "ciudad")
dbutils.fs.cp(
    f"file:{base_path_local}/clientes/clientes_v3.csv",
    f"{base_path_volumes}/clientes/clientes_v3.csv"
)

In [0]:
print("\n🔄 SEGUNDA EVOLUCIÓN - Agregando archivo con otra columna\n")

print("✅ Copiado clientes_v3.csv (incluye columna 'ciudad')")
leer_escribir_csv()

print("\n✅ Segunda evolución completada")


In [0]:
%sql
-- Ver schema completo con todas las evoluciones
SELECT 
    id, 
    nombre, 
    email,
    telefono, 
    ciudad,
    archivo_origen

FROM bronze.clientes 
ORDER BY id

In [0]:
%sql
-- Observar cómo los registros antiguos tienen NULL en columnas nuevas
SELECT 
   *
FROM bronze.clientes
ORDER BY id

---

## 3. Auto Loader con JSON (Ventas)

Ahora trabajaremos con archivos JSON (formato común para APIs y logs).

**Schema inicial (v1):** id, cliente_id, monto, fecha  
**Schema v2:** + producto  
**Schema v3:** + descuento, metodo_pago

---

In [0]:
# Configuración para JSON
json_source = f"{base_path_volumes}/ventas"
json_checkpoint = f"{checkpoint_path}/ventas_json"
json_schema = f"{schema_path}/ventas_json"

# Limpiar checkpoint y schema previos
dbutils.fs.rm(json_checkpoint, True)
dbutils.fs.rm(json_schema, True)
spark.sql("DROP TABLE IF EXISTS bronze.ventas")

print(f"📁 Fuente: {json_source}")
print(f"💾 Checkpoint: {json_checkpoint}")
print(f"📊 Schema: {json_schema}")

In [0]:
from pyspark.sql.functions import year, month, to_date

def leer_escribir_json():
    df_ventas = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", json_schema)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(json_source)
        .withColumn("fecha_ingesta", current_timestamp())
        .withColumn("archivo_origen", col("_metadata.file_path"))
        .withColumn("fecha_date", to_date(col("fecha")))
        .withColumn("year", year(col("fecha_date")))
        .withColumn("mes", month(col("fecha_date")))
    )

    query_ventas = (df_ventas.writeStream
        .format("delta")
        .option("checkpointLocation", json_checkpoint)
        .option("mergeSchema", "true")
        .partitionBy("year", "mes")  # Particionar para optimizar queries temporales
        .trigger(availableNow=True)  # Procesar datos disponibles (compatible con Serverless)
        .table("bronze.ventas")
    )

In [0]:
from pyspark.sql.functions import col, current_timestamp, input_file_name, year, month, to_date

leer_escribir_json()
print("\n✅ Procesamiento inicial completado")

In [0]:
%sql
-- Verificar ventas cargadas (v1)
SELECT *
FROM bronze.ventas 
ORDER BY id

In [0]:
%sql
DESCRIBE EXTENDED bronze.ventas

In [0]:
%sql
-- Ver las particiones físicas creadas
SHOW PARTITIONS bronze.ventas

In [0]:
print("\n🔄 EVOLUCIÓN JSON - Agregando campo 'producto'\n")

# Copiar archivo v2 (con campo "producto")
dbutils.fs.cp(
    f"file:{base_path_local}/ventas/ventas_v2.json",
    f"{base_path_volumes}/ventas/ventas_v2.json"
)

In [0]:
print("✅ Copiado ventas_v2.json (incluye campo 'producto')")
leer_escribir_json()
print("\n✅ Auto Loader detectó y procesó el nuevo schema")

In [0]:
%sql
-- Ver campo 'producto' agregado
SELECT *
FROM bronze.ventas
ORDER BY id

In [0]:
print("\n🔄 SEGUNDA EVOLUCIÓN JSON - Agregando 'descuento' y 'metodo_pago'\n")

# Copiar archivo v3
dbutils.fs.cp(
    f"file:{base_path_local}/ventas/ventas_v3.json",
    f"{base_path_volumes}/ventas/ventas_v3.json"
)

In [0]:


print("✅ Copiado ventas_v3.json (incluye 'descuento' y 'metodo_pago')")
leer_escribir_json()
print("\n✅ Segunda evolución completada")

In [0]:
%sql
-- Schema final con todas las evoluciones
DESCRIBE EXTENDED bronze.ventas

In [0]:
%sql
-- Análisis completo mostrando evolución del schema
SELECT 
    id,
    cliente_id,
    monto,
    producto,       -- NULL en v1
    descuento,      -- NULL en v1 y v2
    metodo_pago,    -- NULL en v1 y v2
    fecha,
    CASE 
        WHEN producto IS NULL THEN 'v1 (schema inicial)'
        WHEN descuento IS NULL THEN 'v2 (+ producto)'
        ELSE 'v3 (+ descuento, metodo_pago)'
    END as version_schema
FROM bronze.ventas
ORDER BY id

---

## 4. Auto Loader con Parquet (Productos)

Formato columnar, ideal para producción (comprimido, rápido, con metadata).

**Schema inicial (v1):** id, nombre, precio  
**Schema v2:** + categoria  
**Schema v3:** + stock, proveedor

---

In [0]:
# Configuración para Parquet
parquet_source = f"{base_path_volumes}/productos"
parquet_checkpoint = f"{checkpoint_path}/productos_parquet"
parquet_schema = f"{schema_path}/productos_parquet"
# Limpiar tabla anterior
spark.sql("DROP TABLE IF EXISTS bronze.productos")

# Limpiar checkpoint y schema previos
dbutils.fs.rm(parquet_checkpoint, True)
dbutils.fs.rm(parquet_schema, True)

print(f"📁 Fuente: {parquet_source}")
print(f"💾 Checkpoint: {parquet_checkpoint}")
print(f"📊 Schema: {parquet_schema}")

In [0]:
def leer_escribir_parquet():
    df_productos = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", parquet_schema)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(parquet_source)
        .withColumn("fecha_ingesta", current_timestamp())
        .withColumn("archivo_origen", col("_metadata.file_path"))
    )
    
    query_productos = (df_productos.writeStream
        .format("delta")
        .option("checkpointLocation", parquet_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)  # Procesar datos disponibles (compatible con Serverless)
        .table("bronze.productos")
    )

In [0]:

leer_escribir_parquet()
print("\n✅ Procesamiento inicial completado")

In [0]:
%sql
-- Verificar productos cargados (v1)
SELECT * FROM bronze.productos ORDER BY id

In [0]:
# Copiar archivos v2 (Parquet es directorio)
dbutils.fs.cp(
    f"file:{base_path_local}/productos/productos_v2.parquet",
    f"{base_path_volumes}/productos/productos_v2.parquet",
    recurse=True)

In [0]:
print("\n🔄 EVOLUCIÓN PARQUET - Agregando columna 'categoria'\n")
print("✅ Copiado productos_v2.parquet (incluye 'categoria')")
leer_escribir_parquet()
print("\n✅ Auto Loader detectó y procesó el nuevo schema")

In [0]:
%sql
-- Ver columna 'categoria' agregada
SELECT *
FROM bronze.productos
ORDER BY id

In [0]:
dbutils.fs.cp(
    f"file:{base_path_local}/productos/productos_v3.parquet",
    f"{base_path_volumes}/productos/productos_v3.parquet",
    recurse=True)

In [0]:
print("\n🔄 SEGUNDA EVOLUCIÓN PARQUET - Agregando 'stock' y 'proveedor'\n")
print("✅ Copiado productos_v3.parquet (incluye 'stock' y 'proveedor')")
leer_escribir_parquet()
print("\n✅ Segunda evolución completada")

In [0]:
%sql
-- Ver todas las columnas evolucionadas
SELECT *
FROM bronze.productos
ORDER BY id

---

## 5. Monitoreo y Métricas

Verificamos el estado de los streams y analizamos métricas de procesamiento.

---

In [0]:
%sql
-- Ver todas las tablas creadas
SHOW TABLES IN bronze

In [0]:
%sql
-- Contar registros en cada tabla
SELECT 'clientes' as tabla, COUNT(*) as registros FROM bronze.clientes
UNION ALL
SELECT 'ventas' as tabla, COUNT(*) as registros FROM bronze.ventas
UNION ALL
SELECT 'productos' as tabla, COUNT(*) as registros FROM bronze.productos

---

### ✅ Resumen parcial

- **Auto Loader (`cloudFiles`)**: ingesta incremental automática de archivos nuevos, funciona con CSV, JSON y Parquet, detecta archivos sin listarlos manualmente y escala a millones de archivos.
- **Schema Evolution**: infiere el schema inicial, detecta columnas nuevas, las agrega a la tabla destino y llena con `NULL` los valores faltantes en registros antiguos (`addNewColumns`).
- **Configuraciones clave**: `cloudFiles.format`, `cloudFiles.schemaLocation`, `cloudFiles.schemaEvolutionMode`, `cloudFiles.inferColumnTypes`, `checkpointLocation`, `mergeSchema`.
- **Buenas prácticas**: rutas persistentes para checkpoints/schemas, particionar por columnas de baja cardinalidad, agregar metadata de ingesta, monitorear métricas y detener streams correctamente.

---

---

## 6. `foreachBatch` - Lógica Personalizada

`foreachBatch` permite ejecutar código personalizado en cada micro-lote:
- UPSERT/MERGE operations (actualizar o insertar)
- Deduplicación
- Validaciones complejas
- Escritura a múltiples destinos
- Logging y auditoría

---

In [0]:
base_path_local

In [0]:
# Crear archivos CSV con actualizaciones y nuevos registros

# Carpeta para datos de actualización
update_path_local = f"{base_path_local}/clientes_updates"
os.makedirs(update_path_local, exist_ok=True)

# BATCH 1: Actualizaciones de clientes existentes + nuevos
csv_update_1 = """id,nombre,email,telefono,ciudad,ultima_compra
1,Ana García ACTUALIZADO,ana.nueva@example.com,555-9999,Madrid,2024-02-15
5,Laura Torres VIP,laura.vip@example.com,555-8888,Barcelona,2024-02-14
20,Nuevo Cliente 1,nuevo1@example.com,555-0020,Sevilla,2024-02-13"""

with open(f"{update_path_local}/clientes_update_batch1.csv", "w") as f:
    f.write(csv_update_1)

print("✅ Creado batch 1: Actualizaciones (Ana, Laura) + Nuevo (id=20)")

# BATCH 2: Más actualizaciones y nuevos
csv_update_2 = """id,nombre,email,telefono,ciudad,ultima_compra
2,Luis Martínez PREMIUM,luis.premium@example.com,555-7777,Valencia,2024-02-16
21,Nuevo Cliente 2,nuevo2@example.com,555-0021,Málaga,2024-02-15
22,Nuevo Cliente 3,nuevo3@example.com,555-0022,Bilbao,2024-02-16"""

with open(f"{update_path_local}/clientes_update_batch2.csv", "w") as f:
    f.write(csv_update_2)

print("✅ Creado batch 2: Actualización (Luis) + Nuevos (id=21,22)")

In [0]:
dbutils.fs.cp(update_path_local, f"{base_path_volumes}/clientes_updates", True)

In [0]:
# Configuración para stream con foreachBatch
update_source = f"{base_path_volumes}/clientes_updates"
update_checkpoint = f"{checkpoint_path}/clientes_upsert"
update_schema = f"{schema_path}/clientes_upsert"

# Limpiar
dbutils.fs.rm(update_checkpoint, True)
dbutils.fs.rm(update_schema, True)

# Crear carpeta en Volumes
dbutils.fs.mkdirs(update_source)

print(f"📁 Fuente: {update_source}")
print(f"💾 Checkpoint: {update_checkpoint}")
print(f"📊 Schema: {update_schema}")

In [0]:
# Stream de actualizaciones
df_updates = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", update_schema)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    .load(update_source)
    .withColumn("fecha_actualizacion", current_timestamp())
)

print("✅ Stream de actualizaciones configurado")

In [0]:
from delta.tables import DeltaTable

# Contador de micro-lotes procesados
batch_counter = 0

def upsert_clientes(df_microlote, batch_id):
    """Ejecuta UPSERT: actualiza si existe, inserta si es nuevo."""
    global batch_counter
    batch_counter += 1
    
    print(f"\n🔄 Procesando micro-lote {batch_id} (#{batch_counter})")
    
    # Verificar si la tabla destino existe
    tabla_existe = spark.catalog.tableExists("bronze.clientes2")
    
    if not tabla_existe:
        # Primera ejecución: crear tabla directamente
        print("   📝 Tabla no existe - creando con datos iniciales")
        df_microlote.write.format("delta").mode("overwrite").saveAsTable("bronze.clientes2")
    else:
        # Tabla existe: hacer MERGE (UPSERT)
        print("   🔀 Ejecutando MERGE (UPSERT)")
        
        delta_table = DeltaTable.forName(spark, "bronze.clientes2")
        
        # MERGE: actualiza si id existe, inserta si es nuevo
        (
            delta_table.alias("T")
            .merge(
                df_microlote.alias("S"),
                "T.id = S.id"  # Condición de match
            )
            .whenMatchedUpdateAll()  # Si existe: actualizar todas las columnas
            .whenNotMatchedInsertAll()  # Si no existe: insertar
            .execute()
        )
        
        print("   ✅ MERGE completado")
    
    # Mostrar resumen
    total_registros = spark.table("bronze.clientes2").count()
    print(f"   📊 Total de registros en tabla: {total_registros}")

print("✅ Función upsert_clientes definida")

In [0]:
# Iniciar stream con foreachBatch
query_upsert = (df_updates.writeStream
    .foreachBatch(upsert_clientes)  # Usar nuestra función personalizada
    .option("checkpointLocation", update_checkpoint)
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)
print("✅ Stream con foreachBatch iniciado")
print("\n⏳ Esperando... (el stream está activo pero no hay archivos aún)")

In [0]:
%sql
-- Estado actual de clientes (antes de actualizaciones)
SELECT 
    id, 
    nombre, 
    email,
    CASE 
        WHEN id <= 5 THEN '⭐ Registro original v1'
        WHEN id <= 10 THEN '⭐ Registro original v2'
        WHEN id <= 15 THEN '⭐ Registro original v3'
        ELSE '🆕 Nuevo'
    END as tipo
FROM bronze.clientes2
ORDER BY id

In [0]:
print("\n🚀 BATCH 1: Copiando actualizaciones...\n")

# Copiar batch 1
dbutils.fs.cp(
    f"file:{update_path_local}/clientes_update_batch1.csv",
    f"{update_source}/clientes_update_batch1.csv"
)

print("✅ Batch 1 copiado")
print("   - Actualizará: Ana (id=1), Laura (id=5)")
print("   - Insertará: Nuevo Cliente 1 (id=20)")
print("\n⏳ Esperando 15 segundos para procesamiento...")

time.sleep(15)

print("\n✅ Batch 1 procesado")

In [0]:
%sql
-- Ver cambios después del UPSERT
-- Ana y Laura deben tener datos ACTUALIZADOS
-- Debe aparecer Nuevo Cliente 1 (id=20)
SELECT 
    id,
    nombre,
    email,
    telefono,
    ciudad,
    ultima_compra,
    CASE 
        WHEN nombre LIKE '%ACTUALIZADO%' OR nombre LIKE '%VIP%' THEN '🔄 ACTUALIZADO'
        WHEN id >= 20 THEN '🆕 NUEVO (UPSERT)'
        ELSE '📌 Original sin cambios'
    END as estado
FROM bronze.clientes
WHERE id IN (1, 2, 3, 4, 5, 20)
ORDER BY id

In [0]:
print("\n🚀 BATCH 2: Copiando más actualizaciones...\n")

# Copiar batch 2
dbutils.fs.cp(
    f"file:{update_path_local}/clientes_update_batch2.csv",
    f"{update_source}/clientes_update_batch2.csv"
)

print("✅ Batch 2 copiado")
print("   - Actualizará: Luis (id=2)")
print("   - Insertará: Nuevos Clientes (id=21,22)")
print("\n⏳ Esperando 15 segundos...")

time.sleep(15)

print("\n✅ Batch 2 procesado")

In [0]:
%sql
-- Estado final después de todos los UPSERTs
SELECT 
    id,
    nombre,
    email,
    ciudad,
    ultima_compra,
    CASE 
        WHEN nombre LIKE '%ACTUALIZADO%' OR nombre LIKE '%PREMIUM%' OR nombre LIKE '%VIP%' THEN '🔄 ACTUALIZADO'
        WHEN id >= 20 THEN '🆕 INSERTADO'
        ELSE '📌 Original'
    END as estado
FROM bronze.clientes
ORDER BY id

---

## 🔍 Ejemplo 2: Deduplicación en Micro-lotes

Caso común: archivos de ingesta contienen duplicados. Queremos mantener solo el registro más reciente por clave.

---

In [0]:


duplicates_path_local = f"{base_path_local}/ventas_duplicadas"
os.makedirs(duplicates_path_local, exist_ok=True)

# Archivo con duplicados (mismo id, diferente timestamp)
ventas_dup = [
    {"id": 100, "cliente_id": 1, "monto": 150.00, "timestamp": "2024-02-01 10:00:00"},  # Primera versión
    {"id": 100, "cliente_id": 1, "monto": 155.00, "timestamp": "2024-02-01 10:05:00"},  # Actualización (más reciente)
    {"id": 101, "cliente_id": 2, "monto": 200.00, "timestamp": "2024-02-01 11:00:00"},
    {"id": 101, "cliente_id": 2, "monto": 210.00, "timestamp": "2024-02-01 11:10:00"},  # Más reciente
    {"id": 102, "cliente_id": 3, "monto": 300.00, "timestamp": "2024-02-01 12:00:00"},  # Sin duplicados
]

with open(f"{duplicates_path_local}/ventas_dup_batch1.json", "w") as f:
    for venta in ventas_dup:
        f.write(json.dumps(venta) + "\n")

print("✅ Creado archivo con duplicados")
print("   - id=100 aparece 2 veces (queremos la más reciente)")
print("   - id=101 aparece 2 veces (queremos la más reciente)")
print("   - id=102 aparece 1 vez (sin duplicados)")

In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# Configuración
dup_source = f"{base_path_volumes}/ventas_duplicadas"
dup_checkpoint = f"{checkpoint_path}/ventas_dedup"
dup_schema = f"{schema_path}/ventas_dedup"

dbutils.fs.rm(dup_checkpoint, True)
dbutils.fs.rm(dup_schema, True)
dbutils.fs.mkdirs(dup_source)

# Limpiar tabla anterior
spark.sql("DROP TABLE IF EXISTS bronze.ventas_dedup")

# Stream
df_ventas_dup = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", dup_schema)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(dup_source)
)

def dedup_and_write(df_batch, batch_id):
    """Deduplicar micro-lote: mantener solo el registro más reciente por id."""
    print(f"\n🔍 Deduplicando micro-lote {batch_id}")
    print(f"   Registros antes de dedup: {df_batch.count()}")
    
    # Crear ventana para particionar por 'id' y ordenar por 'timestamp' DESC
    window_spec = Window.partitionBy("id").orderBy(col("timestamp").desc())
    
    # Agregar número de fila (1 = más reciente)
    df_with_rank = df_batch.withColumn("rank", row_number().over(window_spec))
    
    # Filtrar solo rank=1 (el más reciente)
    df_dedup = df_with_rank.filter(col("rank") == 1).drop("rank")
    
    registros_dedup = df_dedup.count()
    duplicados_removidos = df_batch.count() - registros_dedup
    
    print(f"   Registros después de dedup: {registros_dedup}")
    print(f"   ❌ Duplicados removidos: {duplicados_removidos}")
    
    # Escribir resultados deduplicados
    df_dedup.write.format("delta").mode("append").saveAsTable("bronze.ventas_dedup")
    
    print("   ✅ Escritura completada")

print("✅ Función de deduplicación definida")

In [0]:
# Iniciar nuevo stream
query_dedup = (df_ventas_dup.writeStream
    .foreachBatch(dedup_and_write)
    .option("checkpointLocation", dup_checkpoint)
    .trigger(availableNow=True)
    .start()
)

print("✅ Stream de deduplicación iniciado")

In [0]:
print("\n📥 Copiando archivo con duplicados...\n")

dbutils.fs.cp(
    f"file:{duplicates_path_local}/ventas_dup_batch1.json",
    f"{dup_source}/ventas_dup_batch1.json"
)

print("✅ Archivo copiado (contiene 5 registros, 4 duplicados)")
print("\n⏳ Esperando 15 segundos...")

time.sleep(15)

print("\n✅ Procesamiento completado")

In [0]:
%sql
-- Resultado: solo 3 registros únicos (uno por id)
-- Para id=100 y id=101 debe aparecer solo el registro más reciente
SELECT 
    id,
    cliente_id,
    monto,
    timestamp,
    CASE 
        WHEN id = 100 AND monto = 155.00 THEN '✅ Correcto (versión más reciente)'
        WHEN id = 100 AND monto = 150.00 THEN '❌ ERROR (versión antigua)'
        WHEN id = 101 AND monto = 210.00 THEN '✅ Correcto (versión más reciente)'
        WHEN id = 101 AND monto = 200.00 THEN '❌ ERROR (versión antigua)'
        ELSE '✅ Sin duplicados'
    END as validacion
FROM bronze.ventas_dedup
ORDER BY id, timestamp

---

## 📊 Comparación: Resultados

### Sin deduplicación:
```
id=100: 2 registros (monto: 150.00 y 155.00)
id=101: 2 registros (monto: 200.00 y 210.00)
id=102: 1 registro  (monto: 300.00)
Total: 5 registros
```

### Con deduplicación (foreachBatch):
```
id=100: 1 registro (monto: 155.00 - el más reciente)
id=101: 1 registro (monto: 210.00 - el más reciente)
id=102: 1 registro (monto: 300.00)
Total: 3 registros únicos
```

---

In [0]:
print("\n🛑 Deteniendo todos los streams...\n")

for stream in spark.streams.active:
    print(f"Deteniendo: {stream.id}")
    stream.stop()

print("\n✅ Todos los streams detenidos")
print(f"\n📊 Total de micro-lotes procesados (UPSERT): {batch_counter}")

---

## 7. Resumen y Conclusiones

### ¿Cuándo usar foreachBatch?

**✅ Casos de uso ideales:**
1. **UPSERT/MERGE** - Actualizar registros existentes o insertar nuevos
2. **Deduplicación** - Mantener solo registros únicos por clave
3. **Validación compleja** - Lógica de negocio que requiere todo el micro-lote
4. **Escritura múltiple** - Escribir a varios destinos (Delta, PostgreSQL, S3)
5. **Enriquecimiento** - Joins complejos o lookups contra otras tablas
6. **Logging/Auditoría** - Registrar métricas por cada micro-lote

**❌ Cuándo NO usar foreachBatch:**
- Transformaciones simples que pueden hacerse con `.select()`, `.filter()`
- Escritura directa a una sola tabla Delta (usar `.table()` directamente)
- Operaciones que no necesitan ver el micro-lote completo

### Diferencias clave

```python
# OPCIÓN 1: Escritura directa (más simple)
df.writeStream.table("mi_tabla")  # ✅ Recomendado para casos simples

# OPCIÓN 2: foreachBatch (más control)
df.writeStream.foreachBatch(mi_funcion)  # ✅ Para lógica compleja
```

### Ventajas de foreachBatch
- ✅ Control total sobre cada micro-lote
- ✅ Acceso a la API completa de DataFrames (no solo streaming)
- ✅ Permite operaciones no disponibles en streaming puro (MERGE, múltiples writes)
- ✅ Manejo de errores personalizado

### Desventajas
- ⚠️ Más código para mantener
- ⚠️ Mayor responsabilidad (manejo de errores, idempotencia)
- ⚠️ No hay garantías automáticas de "exactly-once" (debes implementarlas)

### 🎯 Patrón recomendado

```python
def procesar_microlote(df_batch, batch_id):
    # 1. Validar/transformar
    df_limpio = validar_datos(df_batch)
    
    # 2. Deduplicar si es necesario
    df_dedup = deduplicar(df_limpio)
    
    # 3. Enriquecer con datos externos
    df_enriquecido = enriquecer(df_dedup)
    
    # 4. Hacer UPSERT con MERGE
    hacer_merge(df_enriquecido, "mi_tabla")
    
    # 5. Log de métricas
    registrar_metricas(batch_id, df_batch.count())

df.writeStream.foreachBatch(procesar_microlote).start()
```

### 🔑 Conceptos clave de Auto Loader

1. **`cloudFiles.format`**: el formato de tus archivos (csv, json, parquet, etc.)
2. **`cloudFiles.schemaLocation`**: dónde guardar el schema inferido (obligatorio)
3. **`schemaEvolutionMode`**: cómo manejar cambios de schema (`addNewColumns`, `rescue`, `failOnNewColumns`)
4. **`checkpointLocation`**: dónde guardar el progreso del stream (obligatorio en producción)
5. **`mergeSchema`**: permitir evolución de schema en la tabla Delta destino

### 🎯 Ejercicios adicionales sugeridos

1. **Modo Rescue**: configurar `schemaEvolutionMode = "rescue"` y observar cómo se capturan columnas inesperadas en `_rescued_data`
2. **Manejo de errores**: agregar archivos corruptos (JSON mal formado) y separar registros válidos de erróneos usando `_corrupt_record`
3. **Optimización**: usar `OPTIMIZE` y `Z-ORDER` en las tablas Delta para mejorar rendimiento de queries
4. **Watermarks**: experimentar con `withWatermark()` para manejar datos tardíos (late data)

### 📚 Recursos adicionales
- [Auto Loader - Databricks](https://docs.databricks.com/ingestion/auto-loader/index.html)
- [Schema Evolution Guide](https://docs.databricks.com/ingestion/auto-loader/schema.html)
- [Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html)

---

**Próxima unidad:** Optimización de pipelines de streaming y arquitecturas Delta Lake avanzadas